# GEE diversity-index mortality models

Minimal GEE notebook for the diversity-index models only:

1. Full sample: raw model vs model with control variables.
2. Strict honey-only subset: raw model vs model with control variables.

The final output is a single HTML report.

In [1]:
from pathlib import Path
import html
import re
import warnings

import numpy as np
import pandas as pd
from statsmodels.genmod.cov_struct import Exchangeable
from statsmodels.genmod.families import Binomial
from statsmodels.genmod.generalized_estimating_equations import GEE


SURVEY_PATH = Path(r"D:\Amsterdam Uni\BEE\2026\Coding\data\cleaned\2026_surveydata_cleaned_v2.csv")
DIVERSITY_INDEX_PATH = Path(r"D:\Amsterdam Uni\BEE\2026\Toxicity\diversity_location_index_crops.csv")
OUTPUT_DIR = Path(r"D:\Amsterdam Uni\BEE\2026\Toxicity\GEE")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_HTML_PATH = OUTPUT_DIR / "gee_model_diversity_crops_results.html"

RADII_M = [500, 1000]
DIVERSITY_COL = "diversity_index"
CLUSTER_COL = "id_beekeeper"
TRIAL_COL = "2025_winter_in"
DEATH_COL = "2025_winter_dood"
DEATH_COUNT_COL = "death_count"
OUTCOME_COL = "death_rate"
CONTROL_COLUMNS = ["imker_methode", "winter_food", "varroa_treatment"]

RAW_FORMULA = f"{OUTCOME_COL} ~ {DIVERSITY_COL}"
CONTROL_FORMULA = (
    f"{OUTCOME_COL} ~ {DIVERSITY_COL} "
    "+ C(imker_methode) + C(winter_food) + C(varroa_treatment)"
)

In [2]:
def parse_count_value(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    if text in {"", "nan"}:
        return np.nan
    if text in {"nee", "no", "n", "o"}:
        return 0.0
    match = re.search(r"\d+(?:[,.]\d+)?", text)
    if not match:
        return np.nan
    return float(match.group(0).replace(",", "."))


def load_data():
    survey = pd.read_csv(SURVEY_PATH, encoding="latin1")
    diversity_index = pd.read_csv(DIVERSITY_INDEX_PATH)
    return survey, diversity_index


def prepare_survey(survey):
    survey = survey.copy()
    if "delete" in survey.columns:
        survey = survey.loc[survey["delete"].fillna("").astype(str).str.upper().ne("Y")].copy()

    for column in ["id_location", "id_beekeeper", TRIAL_COL, *CONTROL_COLUMNS]:
        survey[column] = pd.to_numeric(survey[column], errors="coerce")

    survey[DEATH_COUNT_COL] = survey[DEATH_COL].map(parse_count_value)
    survey[OUTCOME_COL] = survey[DEATH_COUNT_COL] / survey[TRIAL_COL]

    survey = survey.loc[
        survey["id_location"].notna()
        & survey["id_beekeeper"].notna()
        & survey[TRIAL_COL].notna()
        & survey[DEATH_COUNT_COL].notna()
        & (survey[TRIAL_COL] > 0)
        & survey[OUTCOME_COL].between(0, 1, inclusive="both")
    ].copy()

    survey["id_location"] = survey["id_location"].astype(int)
    survey["id_beekeeper"] = survey["id_beekeeper"].astype(int)
    survey["winter_food_code"] = survey["winter_food"].astype("Int64")
    survey["winter_food_group"] = survey["winter_food_code"].map(
        {1: "sugar_only", 2: "mixed_honey_sugar", 3: "honey_only"}
    )
    return survey


def model_data_for(survey, diversity_index, radius_m, subset=None, required_columns=None):
    required_columns = required_columns or []
    radius_index = diversity_index.loc[
        diversity_index["straal_m"].eq(radius_m),
        ["id_location", DIVERSITY_COL],
    ].copy()

    data = survey.merge(radius_index, on="id_location", how="inner", validate="many_to_one")
    if subset is not None:
        data = data.loc[subset(data)].copy()

    drop_columns = [
        "id_location",
        CLUSTER_COL,
        TRIAL_COL,
        DEATH_COUNT_COL,
        OUTCOME_COL,
        DIVERSITY_COL,
        *required_columns,
    ]
    return data.dropna(subset=drop_columns).copy()

In [3]:
def run_gee(data, formula, label, radius_m):
    if data.empty:
        warnings.warn(f"Skipping {label}: no rows after filtering.")
        return None
    if data[CLUSTER_COL].nunique() < 2:
        warnings.warn(f"Skipping {label}: fewer than 2 beekeeper clusters.")
        return None

    model = GEE.from_formula(
        formula,
        groups=CLUSTER_COL,
        data=data,
        family=Binomial(),
        cov_struct=Exchangeable(),
        weights=data[TRIAL_COL],
    )
    result = model.fit()
    return {
        "label": label,
        "radius_m": radius_m,
        "formula": formula,
        "model_data": data,
        "result": result,
    }


def fit_requested_models(survey, diversity_index):
    outputs = {"Full sample": [], "Strict honey-only": []}
    specs = [
        ("raw", RAW_FORMULA, []),
        ("controls", CONTROL_FORMULA, CONTROL_COLUMNS),
    ]

    for radius_m in RADII_M:
        for name, formula, required_columns in specs:
            full_data = model_data_for(
                survey,
                diversity_index,
                radius_m,
                required_columns=required_columns,
            )
            outputs["Full sample"].append(
                run_gee(full_data, formula, f"full_sample_{name}__{radius_m}m", radius_m)
            )

            honey_data = model_data_for(
                survey,
                diversity_index,
                radius_m,
                subset=lambda df: df["winter_food_group"].eq("honey_only"),
                required_columns=["winter_food_group", *required_columns],
            )
            outputs["Strict honey-only"].append(
                run_gee(honey_data, formula, f"strict_honey_only_{name}__{radius_m}m", radius_m)
            )

    return outputs

In [4]:
def significance_stars(p_value):
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    if p_value < 0.1:
        return "+"
    return ""


def format_coef_cell(output, term):
    result = output["result"]
    if term not in result.params.index:
        return ""
    return f"{result.params[term]:.3f}{significance_stars(result.pvalues[term])}<br>({result.bse[term]:.3f})"


def format_or_cell(output, term=DIVERSITY_COL):
    result = output["result"]
    if term not in result.params.index:
        return ""
    conf_low, conf_high = result.conf_int().loc[term]
    coef = result.params[term]
    return f"{np.exp(coef * 0.1):.3f} [{np.exp(conf_low * 0.1):.3f}, {np.exp(conf_high * 0.1):.3f}]"


def ordered_terms(outputs):
    terms = []
    for output in outputs:
        if output is None:
            continue
        for term in output["result"].params.index:
            if term not in terms:
                terms.append(term)
    preferred = ["Intercept", DIVERSITY_COL]
    return [term for term in preferred if term in terms] + [term for term in terms if term not in preferred]


def make_results_table(outputs):
    outputs = [output for output in outputs if output is not None]
    rows = []
    for term in ordered_terms(outputs):
        row = {"Term": term}
        for output in outputs:
            row[output["label"]] = format_coef_cell(output, term)
        rows.append(row)

    info_rows = [
        ("Locations", lambda output: str(len(output["model_data"]))),
        ("Colonies entering winter", lambda output: str(int(output["model_data"][TRIAL_COL].sum()))),
        ("Observed deaths", lambda output: str(int(output["model_data"][DEATH_COUNT_COL].sum()))),
        ("Beekeeper clusters", lambda output: str(output["model_data"][CLUSTER_COL].nunique())),
        ("OR per 0.1 diversity_index [95% CI]", format_or_cell),
        ("p-value, diversity_index", lambda output: f"{output['result'].pvalues[DIVERSITY_COL]:.3f}"),
    ]
    for label, getter in info_rows:
        row = {"Term": label}
        for output in outputs:
            row[output["label"]] = getter(output)
        rows.append(row)
    return pd.DataFrame(rows)


def save_html_report(sections):
    html_sections = []
    for title, outputs in sections.items():
        outputs = [output for output in outputs if output is not None]
        if not outputs:
            continue
        table = make_results_table(outputs)
        html_sections.append(f"<h2>{html.escape(title)}</h2>\n{table.to_html(index=False, escape=False)}")

    html_doc = f"""<!doctype html>
<html>
<head>
  <meta charset="utf-8">
  <title>GEE diversity-index mortality models</title>
  <style>
    body {{ font-family: Arial, sans-serif; margin: 32px; line-height: 1.35; color: #24292f; }}
    h1 {{ margin-bottom: 8px; }}
    h2 {{ margin-top: 32px; }}
    table {{ border-collapse: collapse; font-size: 14px; margin-bottom: 24px; }}
    th, td {{ border: 1px solid #d0d7de; padding: 8px 10px; vertical-align: top; }}
    th {{ background: #f6f8fa; }}
    tr:nth-child(even) {{ background: #fbfbfb; }}
    td:first-child, th:first-child {{ font-weight: 600; min-width: 230px; }}
  </style>
</head>
<body>
  <h1>GEE diversity-index mortality models</h1>
  <p>Outcome: death_rate = 2025_winter_dood / 2025_winter_in.</p>
  <p>Models: Binomial GEE, exchangeable working correlation, clustered by id_beekeeper, weighted by 2025_winter_in.</p>
  <p>Cells show coefficient with robust standard error in parentheses. Significance: + p&lt;0.10, * p&lt;0.05, ** p&lt;0.01, *** p&lt;0.001.</p>
  {''.join(html_sections)}
</body>
</html>
"""
    RESULTS_HTML_PATH.write_text(html_doc, encoding="utf-8")
    return RESULTS_HTML_PATH

In [5]:
survey_raw, diversity_index = load_data()
survey = prepare_survey(survey_raw)
model_sections = fit_requested_models(survey, diversity_index)
save_html_report(model_sections)

WindowsPath('D:/Amsterdam Uni/BEE/2026/Toxicity/GEE/gee_model_diversity_crops_results.html')